In [ ]:
def main(datasources, start_date, end_date):
    """C03: depth slope asymmetry x overnight gap repair (standalone)."""
    import numpy as np
    import pandas as pd
    import dai

    table = (datasources or {}).get("bar1m", "bigalpha_2026_stock_bar1m")
    sql = f"""
    WITH m AS (
      SELECT date, instrument, open, close, pre_close, high, low, amount, volume,
        deal_number,
        COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0) AS bd3,
        COALESCE(ask_volume1,0)+COALESCE(ask_volume2,0)+COALESCE(ask_volume3,0) AS ad3,
        COALESCE(bid_volume4,0)+COALESCE(bid_volume5,0) AS bd45,
        COALESCE(ask_volume4,0)+COALESCE(ask_volume5,0) AS ad45,
        (CASE WHEN COALESCE(bid_volume1,0)+COALESCE(ask_volume1,0)>0 THEN
          (COALESCE(ask_price1,close)*COALESCE(bid_volume1,0)+COALESCE(bid_price1,close)*COALESCE(ask_volume1,0)) /
          (COALESCE(bid_volume1,0)+COALESCE(ask_volume1,0)) ELSE close END-close)/NULLIF(close,0) AS micro_gap
      FROM {table}
    ), d AS (
      SELECT date_trunc('day',date)::DATE AS date, instrument,
        ARG_MIN(open,date) AS day_open, ARG_MAX(close,date) AS day_close,
        MAX(pre_close) AS pre_close, MAX(high) AS day_high, MIN(low) AS day_low,
        (MAX(high)-MIN(low))/NULLIF(AVG(close),0) AS range_ratio,
        SUM(amount) AS amount_sum, STDDEV_POP(amount)/NULLIF(AVG(amount)+1,0) AS amount_concentration,
        SUM(volume) AS volume_sum, SUM(deal_number) AS deal_number_sum,
        AVG(bd3) AS bd3, AVG(ad3) AS ad3, AVG(bd45) AS bd45, AVG(ad45) AS ad45,
        AVG(micro_gap) AS micro_gap
      FROM m GROUP BY date_trunc('day',date)::DATE, instrument
    ) SELECT * FROM d ORDER BY date,instrument
    """
    x = dai.query(sql, filters={"date": [start_date, end_date]}, compression=True).df()
    x["date"] = pd.to_datetime(x["date"])
    x["instrument"] = x["instrument"].astype(str)
    numeric = [c for c in x.columns if c not in ("date", "instrument")]
    x[numeric] = x[numeric].apply(pd.to_numeric, errors="coerce")

    def rank_z(s):
        r = s.replace([np.inf, -np.inf], np.nan).rank(pct=True, method="average")
        r = r - r.mean()
        sd = r.std(ddof=0)
        return r * 0.0 if not np.isfinite(sd) or sd <= 1e-12 else r / sd

    def rz(s):
        return s.groupby(x["date"]).transform(rank_z)

    intraday = x["day_close"] / x["day_open"].replace(0, np.nan) - 1
    gap = x["day_open"] / x["pre_close"].replace(0, np.nan) - 1
    absorption = -gap * intraday
    depth_slope = np.log1p(x["bd3"]/(x["bd45"]+1)) - np.log1p(x["ad3"]/(x["ad45"]+1))
    gap_repair = rz(absorption) + 0.5 * rz(intraday) - 0.35 * rz(gap).abs()
    raw = rz(depth_slope) * rz(gap_repair)
    x["factor"] = rz(raw).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    pool = dai.query("SELECT date,instrument FROM bigalpha_2026_instruments",
                     filters={"date": [start_date, end_date]}, compression=True).df()
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    out = pool.merge(x[["date","instrument","factor"]], on=["date","instrument"], how="left")
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(0.0)
    return out[["date","instrument","factor"]].reset_index(drop=True)
